In [ ]:
# Required imports
import os
import cv2
import numpy as np
import tensorflow as tf
from keras.utils import img_to_array
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras.optimizers import Adam

# Constants
SEQUENCE_LENGTH = 20
IMAGE_HEIGHT = 64
IMAGE_WIDTH = 64
CHANNELS = 3
BATCH_SIZE = 16
EPOCHS = 50

# Dataset Loader
class VideoDataset:
    def __init__(self, data_path, sequence_length, image_size):
        self.data_path = data_path
        self.sequence_length = sequence_length
        self.image_size = image_size
    
    def load_video_frames(self, video_path):
        video_path = video_path.numpy().decode('utf-8')
        cap = cv2.VideoCapture(video_path)
        frames = []
        while len(frames) < self.sequence_length:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.resize(frame, self.image_size)
            frame = img_to_array(frame) / 255.0
            frames.append(frame)
        cap.release()
        frames = frames[:self.sequence_length]
        while len(frames) < self.sequence_length:
            frames.append(np.zeros((*self.image_size, CHANNELS)))
        return np.array(frames, dtype=np.float32)

# Model Architectures
def create_enhanced_convlstm_model(seq_length, height, width, channels, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(32, (3,3), activation='relu', padding='same'), input_shape=(seq_length, height, width, channels)),
        tf.keras.layers.TimeDistributed(tf.keras.layers.MaxPooling2D(2,2)),
        tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same')),
        tf.keras.layers.TimeDistributed(tf.keras.layers.MaxPooling2D(2,2)),
        tf.keras.layers.TimeDistributed(tf.keras.layers.Flatten()),
        tf.keras.layers.LSTM(64, return_sequences=False),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

def create_enhanced_lrcn_model(seq_length, height, width, channels, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.ConvLSTM2D(32, (3,3), activation='relu', padding='same', return_sequences=True, input_shape=(seq_length, height, width, channels)),
        tf.keras.layers.ConvLSTM2D(64, (3,3), activation='relu', padding='same', return_sequences=False),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Model Trainer
class ModelTrainer:
    def __init__(self, model_type, num_classes):
        self.strategy = tf.distribute.MirroredStrategy()
        with self.strategy.scope():
            self.model_type = model_type
            self.num_classes = num_classes
            self.model = self.build_model()
    
    def build_model(self):
        if self.model_type == 'convlstm':
            return create_enhanced_convlstm_model(SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH, CHANNELS, self.num_classes)
        else:
            return create_enhanced_lrcn_model(SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH, CHANNELS, self.num_classes)
    
    def compile_model(self):
        with self.strategy.scope():
            optimizer = Adam(learning_rate=0.001)
            self.model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), tf.keras.metrics.AUC()])
    
    def train_and_validate(self, train_dataset, val_dataset):
        history = self.model.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS)
        return history

# Model Evaluator
class ModelEvaluator:
    @staticmethod
    def evaluate_model(model, test_dataset, label_encoder):
        y_true, y_pred = [], []
        for x, y in test_dataset:
            preds = model.predict(x)
            y_pred.extend(np.argmax(preds, axis=1))
            y_true.extend(np.argmax(y.numpy(), axis=1))
        report = classification_report(y_true, y_pred, target_names=label_encoder.classes_, output_dict=True)
        return report

# Main Function
def main():
    data_path = 'my_dataset/'
    video_paths, labels = [], []
    for category in ['Normal', 'Abnormal']:
        category_path = os.path.join(data_path, category)
        if os.path.exists(category_path):
            for video_file in os.listdir(category_path):
                if video_file.endswith(('.mp4', '.avi')):
                    video_paths.append(os.path.join(category_path, video_file))
                    labels.append(category)
    
    # Convert lists to numpy arrays (Fix applied)
    video_paths = np.array(video_paths, dtype=object)  # Ensure NumPy indexing works
    one_hot_labels = np.array(tf.keras.utils.to_categorical(LabelEncoder().fit_transform(labels), len(set(labels))))
    dataset = VideoDataset(data_path, SEQUENCE_LENGTH, (IMAGE_HEIGHT, IMAGE_WIDTH))
    
    def load_and_preprocess(video_path, label):
        frames = tf.py_function(func=dataset.load_video_frames, inp=[video_path], Tout=tf.float32)
        frames.set_shape((SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH, CHANNELS))
        return frames, label
    
    indices = np.random.permutation(len(video_paths))
    train_size, val_size = int(0.7 * len(indices)), int(0.15 * len(indices))
    train_indices, val_indices, test_indices = indices[:train_size], indices[train_size:train_size+val_size], indices[train_size+val_size:]
    
    train_dataset = tf.data.Dataset.from_tensor_slices((video_paths[train_indices], one_hot_labels[train_indices])) \
        .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE) \
        .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
    val_dataset = tf.data.Dataset.from_tensor_slices((video_paths[val_indices], one_hot_labels[val_indices])) \
        .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE) \
        .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
    test_dataset = tf.data.Dataset.from_tensor_slices((video_paths[test_indices], one_hot_labels[test_indices])) \
        .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE) \
        .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
    for model_type in ['convlstm', 'lrcn']:
        trainer = ModelTrainer(model_type, len(set(labels)))
        trainer.compile_model()
        history = trainer.train_and_validate(train_dataset, val_dataset)
        evaluator = ModelEvaluator()
        report = evaluator.evaluate_model(trainer.model, test_dataset, LabelEncoder().fit(labels))
        print(f"{model_type.upper()} Model Report:", report)
        trainer.model.save(f'{model_type}_final.h5')

if __name__ == "__main__":
    main()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0',)


/mnt/g/Study/Academic/Part 4/Project/.venv/lib/python3.12/site-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50


/mnt/g/Study/Academic/Part 4/Project/.venv/lib/python3.12/site-packages/keras/src/ops/nn.py:907: UserWarning: You are using a softmax over axis -1 of a tensor of shape (16, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
/mnt/g/Study/Academic/Part 4/Project/.venv/lib/python3.12/site-packages/keras/src/losses/losses.py:33: SyntaxWarning: In loss categorical_crossentropy, expected y_pred.shape to be (batch_size, num_classes) with num_classes > 1. Received: y_pred.shape=(16, 1). Consider using 'binary_crossentropy' if you only have 2 classes.
  return self.fn(y_true, y_pred, **self._fn_kwargs)
/mnt/g/Study/Academic/Part 4/Project/.venv/lib/python3.12/site-packages/keras/src/ops/nn.py:907: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not 

36/36 ━━━━━━━━━━━━━━━━━━━━ 23s 437ms/step - accuracy: 1.0000 - auc_4: 0.0000e+00 - loss: 0.0000e+00 - precision_4: 1.0000 - recall_4: 1.0000 - val_accuracy: 1.0000 - val_auc_4: 0.0000e+00 - val_loss: 0.0000e+00 - val_precision_4: 1.0000 - val_recall_4: 1.0000
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 13s 345ms/step - accuracy: 1.0000 - auc_4: 0.0000e+00 - loss: 0.0000e+00 - precision_4: 1.0000 - recall_4: 1.0000 - val_accuracy: 1.0000 - val_auc_4: 0.0000e+00 - val_loss: 0.0000e+00 - val_precision_4: 1.0000 - val_recall_4: 1.0000
Epoch 3/50


...

36/36 ━━━━━━━━━━━━━━━━━━━━ 13s 356ms/step - accuracy: 1.0000 - auc_4: 0.0000e+00 - loss: 0.0000e+00 - precision_4: 1.0000 - recall_4: 1.0000 - val_accuracy: 1.0000 - val_auc_4: 0.0000e+00 - val_loss: 0.0000e+00 - val_precision_4: 1.0000 - val_recall_4: 1.0000
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 15s 413ms/step - accuracy: 1.0000 - auc_4: 0.0000e+00 - loss: 0.0000e+00 - precision_4: 1.0000 - recall_4: 1.0000 - val_accuracy: 1.0000 - val_auc_4: 0.0000e+00 - val_loss: 0.0000e+00 - val_precision_4: 1.0000 - val_recall_4: 1.0000
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 12s 342ms/step - accuracy: 1.0000 - auc_4: 0.0000e+00 - loss: 0.0000e+00 - precision_4: 1.0000 - recall_4: 1.0000 - val_accuracy: 1.0000 - val_auc_4: 0.0000e+00 - val_loss: 0.0000e+00 - val_precision_4: 1.0000 - val_recall_4: 1.0000
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 15s 415ms/step - accuracy: 1.0000 - auc_4: 0.0000e+00 - loss: 0.0000e+00 - precision_4: 1.0000 - recall_4: 1.0000 - val_accuracy: 1.0000 - val_auc_4: 0.0000e+00

2025-02-14 14:16:53.396565: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node MultiDeviceIteratorGetNextFromShard}}]]
	 [[RemoteCall]]


36/36 ━━━━━━━━━━━━━━━━━━━━ 14s 376ms/step - accuracy: 1.0000 - auc_4: 0.0000e+00 - loss: 0.0000e+00 - precision_4: 1.0000 - recall_4: 1.0000 - val_accuracy: 1.0000 - val_auc_4: 0.0000e+00 - val_loss: 0.0000e+00 - val_precision_4: 1.0000 - val_recall_4: 1.0000
Epoch 9/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 17s 465ms/step - accuracy: 1.0000 - auc_4: 0.0000e+00 - loss: 0.0000e+00 - precision_4: 1.0000 - recall_4: 1.0000 - val_accuracy: 1.0000 - val_auc_4: 0.0000e+00 - val_loss: 0.0000e+00 - val_precision_4: 1.0000 - val_recall_4: 1.0000
Epoch 10/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 13s 353ms/step - accuracy: 1.0000 - auc_4: 0.0000e+00 - loss: 0.0000e+00 - precision_4: 1.0000 - recall_4: 1.0000 - val_accuracy: 1.0000 - val_auc_4: 0.0000e+00 - val_loss: 0.0000e+00 - val_precision_4: 1.0000 - val_recall_4: 1.0000
Epoch 11/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 16s 449ms/step - accuracy: 1.0000 - auc_4: 0.0000e+00 - loss: 0.0000e+00 - precision_4: 1.0000 - recall_4: 1.0000 - val_accuracy: 1.0000 - val_auc_4: 0.0000e+

2025-02-14 14:19:20.481448: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node MultiDeviceIteratorGetNextFromShard}}]]
	 [[RemoteCall]]


36/36 ━━━━━━━━━━━━━━━━━━━━ 13s 356ms/step - accuracy: 1.0000 - auc_4: 0.0000e+00 - loss: 0.0000e+00 - precision_4: 1.0000 - recall_4: 1.0000 - val_accuracy: 1.0000 - val_auc_4: 0.0000e+00 - val_loss: 0.0000e+00 - val_precision_4: 1.0000 - val_recall_4: 1.0000
Epoch 20/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 13s 346ms/step - accuracy: 1.0000 - auc_4: 0.0000e+00 - loss: 0.0000e+00 - precision_4: 1.0000 - recall_4: 1.0000 - val_accuracy: 1.0000 - val_auc_4: 0.0000e+00 - val_loss: 0.0000e+00 - val_precision_4: 1.0000 - val_recall_4: 1.0000
Epoch 21/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 16s 430ms/step - accuracy: 1.0000 - auc_4: 0.0000e+00 - loss: 0.0000e+00 - precision_4: 1.0000 - recall_4: 1.0000 - val_accuracy: 1.0000 - val_auc_4: 0.0000e+00 - val_loss: 0.0000e+00 - val_precision_4: 1.0000 - val_recall_4: 1.0000
Epoch 22/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 13s 354ms/step - accuracy: 1.0000 - auc_4: 0.0000e+00 - loss: 0.0000e+00 - precision_4: 1.0000 - recall_4: 1.0000 - val_accuracy: 1.0000 - val_auc_4: 0.0000e